# 5.12 改幾個字就算新資料嗎？


「紅色圓形在左邊」與「紅色圓形在右邊」只差一字，完整指紋不同，仍是相近資料。為避免幾乎同題的版本跨訓練與測試，再看文字重疊。

各取連續兩字片段，共有紅色、色圓、圓形、形在四種，聯集八種，比例4/8=0.5。共有集合大小除以聯集大小叫Jaccard相似度。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
def grams(text, n=2):
    return {text[i : i + n] for i in range(len(text) - n + 1)}


a = grams("紅色圓形在左邊")
b = grams("紅色圓形在右邊")
common = a & b
all_parts = a | b
score = len(common) / len(all_parts)
print("共有片段", sorted(common))
print("共有數", len(common), "聯集數", len(all_parts), "相似度", score)

共有片段 ['圓形', '形在', '紅色', '色圓']
共有數 4 聯集數 8 相似度 0.5


輸入兩段七字的句子，`grams` 預設n=2，收集每個起點的連續兩字。i從0到5，共六段；右端 `i+n` 不包含，所以每次正好取兩字。大括號內的迴圈建立集合，重複片段只保留一份。`&` 取兩集合交集，`|` 取聯集，清單/集合背景可回看 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2) 與 [1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html)。

輸出共有四段，聯集八段，相似度0.5；`sorted` 只讓顯示順序固定，不參與計分。這裡兩字片段常叫character n-gram，n就是片段長度，它與訓練窗口不同：片段用來比較文字重疊，訓練窗口是模型實際接收的前文。這個小函數假定非空且長度至少n，若兩集合都空，分母0必須另定義處理規則。

相近不等於同義：左與右正是關鍵答案，不能把它們合成同一標籤。可以將相近版本放同家族切分，仍保留各自正解。片段法會漏不同措辭的同義句，也可能誤合短句；家族判定還需來源與改寫記錄。門檻要連同片段長度保存，沒有通用0.5標準。

練習將兩次grams呼叫都加 `n=3`，先手數每句五個三字片段，共有「紅色圓、色圓形、圓形在」三段，聯集七段，分數3/7約0.429，再執行核對。只改變片段長就改變分數，說明門檻要和比較方式一起定義。

<details>
<summary>補充：實作約定與原始紀錄</summary>

目前[T.4的故事與詩文實跑](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)只做上一節的完整內容去重，還沒有把近重複版本聚成家族。因此「沒有完整重複跨組」可以核對，「所有相似版本都已隔開」則不能宣稱。這份留出成績適合檢查小型訓練流程，不能據此排除模型從相似材料得到提示的可能。[原始報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/real_text.json)保留這項限制；想進一步研究泛化，下一步應先補家族判定，再固定新切分做比較。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
